In [4]:
# https://pypi.org/project/azure-ai-evaluation/

In [ ]:
from azure.ai.evaluation import F1ScoreEvaluator, BleuScoreEvaluator, GleuScoreEvaluator, RougeScoreEvaluator, RougeType, MeteorScoreEvaluator
import json 

with open("llm_explanations/xaiqb_human_verbose.json") as f:
    xaiqb_human = json.load(f)

with open("llm_explanations/xaiqb_opus_1.1_v2.json") as f:
    xaiqb_claude = json.load(f)

with open("llm_explanations/xaiqb_gemini_1.1_v2.json") as f:
    xaiqb_gemini = json.load(f)

In [ ]:
from transformers import pipeline
nli = pipeline("text-classification", model="roberta-large-mnli", top_k=None)

def nli_label(premise, hypothesis):
    # premise entails hypothesis? (using MNLI framing)
    # We return the best label among entailment/neutral/contradiction
    out = nli({"text": premise, "text_pair": hypothesis})[0]
    return out["label"], float(out["score"])

# Example scoring for one Q/A pair
def score_pair(pred, ref):
    entail_ph, conf_ph = nli_label(pred, ref)  # does pred entail ref?
    entail_hp, conf_hp = nli_label(ref, pred)  # does ref entail pred?

In [6]:
f1_score        = F1ScoreEvaluator(threshold = 0.5)
bleu_score      = BleuScoreEvaluator(threshold = 0.3)
gleu_score      = GleuScoreEvaluator(threshold = 0.2)
rouge_score     = RougeScoreEvaluator(rouge_type = RougeType.ROUGE_L, precision_threshold = 0.6, recall_threshold = 0.5, f1_score_threshold = 0.55) 
meteor_score    = MeteorScoreEvaluator(threshold = 0.9)

In [ ]:
import pandas as pd
def text_scores(response, ground_truth):
    text_scores_df = pd.DataFrame({
        "f1": [f1_score(response = response, ground_truth = ground_truth)["f1_score"]],
        "bleu_score": [bleu_score(response = response, ground_truth = ground_truth)["bleu_score"]],
        "gleu_score": [gleu_score(response = response, ground_truth = ground_truth)["gleu_score"]],
        "rouge": [rouge_score(response = response, ground_truth = ground_truth)["rouge_f1_score"]],
        "meteor_score": [meteor_score(response = response, ground_truth = ground_truth)["meteor_score"]],
    })
    return text_scores_df.round(3)

In [8]:
all_scores = []

for question in xaiqb_human["questions"]["Data"].keys():
    all_scores += [text_scores(
        response = xaiqb_claude["questions"]["Data"][question]["answer"],
        ground_truth = xaiqb_human["questions"]["Data"][question]["core_answer"]
        )]

all_scores = pd.concat(all_scores)
all_scores.mean()

f1              0.234333
bleu_score      0.062667
gleu_score      0.078500
rouge           0.184667
meteor_score    0.346167
dtype: float64

In [9]:
all_scores = []

for question in xaiqb_human["questions"]["Data"].keys():
    all_scores += [text_scores(
        response = xaiqb_gemini["questions"]["Data"][question]["answer"],
        ground_truth = xaiqb_human["questions"]["Data"][question]["core_answer"]
        )]

all_scores = pd.concat(all_scores)
all_scores.mean()

f1              0.200167
bleu_score      0.023833
gleu_score      0.062167
rouge           0.181000
meteor_score    0.253167
dtype: float64